In [10]:
import os
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/agropulse_xai/data"
os.makedirs(DATA_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import ee
ee.Authenticate()
ee.Initialize(project='sebastian-borja-spaceapps')

In [ ]:
import os
import json
import requests
import numpy as np
import pandas as pd
import ee
from datetime import datetime

# Inicializar Google Earth Engine (asegúrate de haber corrido ee.Authenticate() antes)
ee.Initialize(project='sebastian-borja-spaceapps')

# ==============================================================================
# 1. CONFIGURACIÓN DEL POLÍGONO CORREGIDO (ZONA AGRÍCOLA STORY COUNTY, IOWA)
# ==============================================================================
CENTER_LON = -93.5150
CENTER_LAT = 42.0830
HALF_SIZE_DEG = 0.0030  # ~600m x 600m (~36 ha, ~400 píxeles a 30m)
SEED = 42

BBOX = [
    CENTER_LON - HALF_SIZE_DEG,
    CENTER_LAT - HALF_SIZE_DEG,
    CENTER_LON + HALF_SIZE_DEG,
    CENTER_LAT + HALF_SIZE_DEG,
]
ROI = ee.Geometry.Rectangle(BBOX)

# Códigos CDL agrícolas válidos (Maíz=1, Soya=5, Cereales/Forrajes/Barbecho)
VALID_AG_CODES = [1, 4, 5, 21, 22, 23, 24, 27, 28, 36, 37, 61, 176]

os.makedirs("data", exist_ok=True)



In [ ]:
# ==============================================================================
# 2. EXTRACCIÓN DE SUELO REAL DESDE USDA SSURGO (SDA REST API)
# ==============================================================================
def fetch_ssurgo_soil(bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    wkt = (
        f"POLYGON(({min_lon} {min_lat}, {max_lon} {min_lat}, "
        f"{max_lon} {max_lat}, {min_lon} {max_lat}, {min_lon} {min_lat}))"
    )
    query = f"""
    SELECT
        mu.mukey,
        SUM(c.comppct_r * ch.awc_r) / NULLIF(SUM(c.comppct_r), 0) AS awc,
        SUM(c.comppct_r * CAST(ch.kwfact AS FLOAT)) / NULLIF(SUM(c.comppct_r), 0) AS k_factor,
        SUM(c.comppct_r * ch.om_r) / NULLIF(SUM(c.comppct_r), 0) AS organic_matter,
        SUM(c.comppct_r * ch.sandtotal_r) / NULLIF(SUM(c.comppct_r), 0) AS sand_pct,
        SUM(c.comppct_r * ch.claytotal_r) / NULLIF(SUM(c.comppct_r), 0) AS clay_pct
    FROM mapunit mu
    INNER JOIN component c ON c.mukey = mu.mukey AND c.majcompflag = 'Yes'
    INNER JOIN chorizon ch ON ch.cokey = c.cokey AND ch.hzdept_r < 100
    WHERE mu.mukey IN (
        SELECT DISTINCT mukey FROM SDA_Get_Mukey_from_intersection_with_WktWgs84('{wkt}')
    )
    GROUP BY mu.mukey
    """
    url = "https://sdmdataaccess.sc.egov.usda.gov/Tabular/post.rest"
    resp = requests.post(url, json={"query": query, "format": "JSON"}, timeout=30)
    resp.raise_for_status()
    table = resp.json().get("Table", [])
    if not table:
        raise ValueError("SSURGO no devolvió datos edáficos para este polígono.")

    df_soil = pd.DataFrame(
        table,
        columns=["mukey", "awc", "k_factor", "organic_matter", "sand_pct", "clay_pct"],
    ).astype(float)
    # Promedio ponderado de las unidades cartográficas del lote
    return df_soil.mean().to_dict()



In [ ]:
# ==============================================================================
# 3. GRILLA ESTÁTICA 30M (CDL 2016-2025 + PENDIENTE SRTM + SSURGO)
# ==============================================================================
def build_grid_30m():
    print("Extrayendo CDL (2016-2025) y SRTM en GEE...")
    years = list(range(2016, 2026))
    cdl_imgs = []
    for y in years:
        img = (
            ee.ImageCollection("USDA/NASS/CDL")
            .filter(ee.Filter.calendarRange(y, y, "year"))
            .first()
            .select(["cropland"], [f"cdl_{y}"])
        )
        cdl_imgs.append(img)

    srtm = ee.Image("USGS/SRTMGL1_003")
    slope = ee.Terrain.slope(srtm).rename("slope_deg")
    coords = ee.Image.pixelLonLat()

    stack = ee.Image.cat(cdl_imgs + [slope, coords])
    samples = stack.sample(region=ROI, scale=30, geometries=False).getInfo()

    rows = [f["properties"] for f in samples["features"]]
    df = pd.DataFrame(rows).rename(columns={"longitude": "lon", "latitude": "lat"})

    # Auditoría anti-urbana (Compuerta de calidad)
    total_px = len(df)
    is_ag = df["cdl_2024"].isin(VALID_AG_CODES) & df["cdl_2025"].isin(VALID_AG_CODES)
    ag_ratio = is_ag.mean()

    print(f"Auditoría CDL: {total_px} píxeles totales | {ag_ratio*100:.1f}% agrícolas válidos.")
    if ag_ratio < 0.75:
        raise RuntimeError(
            f"ALERTA: Solo {ag_ratio*100:.1f}% de píxeles son agrícolas. Revisa las coordenadas."
        )

    df = df[is_ag].reset_index(drop=True)
    df.insert(0, "pixel_id", [f"px_{i:04d}" for i in range(len(df))])

    # Cruzar con propiedades edáficas reales de SSURGO
    soil_props = fetch_ssurgo_soil(BBOX)
    for col in ["awc", "k_factor", "organic_matter", "sand_pct", "clay_pct"]:
        df[col] = round(soil_props[col], 4)

    df.to_parquet("data/grid_30m.parquet", index=False)
    print(f"Guardado: data/grid_30m.parquet ({len(df)} píxeles limpios).")
    return df



In [ ]:

# ==============================================================================
# 4. SERIES ÓPTICAS HLS (2016-2025) CON getRegion POR QUINCENA (< 400K VALORES)
# ==============================================================================
def build_hls_series(df_grid):
    print("Extrayendo series HLS (Landsat + Sentinel armonizado) por quincenas...")
    l30 = ee.ImageCollection("NASA/HLS/HLSL30/v002").select(
        ["B2", "B3", "B4", "B5", "B6", "B7", "Fmask"],
        ["blue", "green", "red", "nir", "swir1", "swir2", "fmask"],
    )
    s30 = ee.ImageCollection("NASA/HLS/HLSS30/v002").select(
        ["B2", "B3", "B4", "B8A", "B11", "B12", "Fmask"],
        ["blue", "green", "red", "nir", "swir1", "swir2", "fmask"],
    )
    hls = l30.merge(s30).filterBounds(ROI)

    grid_coords = df_grid[["lon", "lat"]].values
    grid_ids = df_grid["pixel_id"].values
    cols_keep = ["pixel_id", "date", "blue", "green", "red", "nir", "swir1", "swir2", "fmask"]

    all_dfs = []
    for y in range(2016, 2026):
        year_obs = 0
        for m in range(1, 13):
            d_start = ee.Date.fromYMD(y, m, 1)
            d_mid = ee.Date.fromYMD(y, m, 16)
            d_end = d_start.advance(1, "month")

            for p_start, p_end in [(d_start, d_mid), (d_mid, d_end)]:
                col_p = hls.filterDate(p_start, p_end)
                try:
                    raw = col_p.getRegion(ROI, 30).getInfo()
                except ee.EEException:
                    continue

                if not raw or len(raw) <= 1:
                    continue

                df_p = pd.DataFrame(raw[1:], columns=raw[0]).dropna(subset=["nir"]).copy()
                if df_p.empty:
                    continue

                pts = df_p[["longitude", "latitude"]].values
                dists = np.linalg.norm(pts[:, None, :] - grid_coords[None, :, :], axis=2)
                min_idx = np.argmin(dists, axis=1)
                min_dist = dists[np.arange(len(pts)), min_idx]

                df_p["pixel_id"] = grid_ids[min_idx]
                df_p = df_p[min_dist < 0.0005].copy()
                df_p["date"] = pd.to_datetime(df_p["time"], unit="ms").dt.normalize()

                all_dfs.append(df_p[cols_keep])
                year_obs += len(df_p)

        print(f"  Año {y}: {year_obs} observaciones válidas.")

    df_hls = pd.concat(all_dfs, ignore_index=True)
    # Promediar traslapes de órbita del mismo día para entregar 1 fila limpia por píxel/día
    df_hls = (
        df_hls.groupby(["pixel_id", "date"], as_index=False)
        .mean(numeric_only=True)
        .sort_values(["pixel_id", "date"])
        .reset_index(drop=True)
    )
    df_hls["fmask"] = df_hls["fmask"].round().astype("Int64")
    df_hls.to_parquet("data/hls_series_2016_2025.parquet", index=False)
    print(f"Guardado: data/hls_series_2016_2025.parquet ({len(df_hls)} filas).")


In [ ]:
import shutil

# ==============================================================================
# BLOQUE FINAL DEFINITIVO: CLIMA + SMAP/GPM + AUDITORÍA + ZIP
# ==============================================================================
def compute_et0_hargreaves(df, lat_deg):
    doy = df["date"].dt.dayofyear.values
    lat_rad = np.deg2rad(lat_deg)
    dr = 1 + 0.033 * np.cos(2 * np.pi / 365 * doy)
    delta = 0.409 * np.sin(2 * np.pi / 365 * doy - 1.39)
    ws = np.arccos(-np.tan(lat_rad) * np.tan(delta))
    ra = (24 * 60 / np.pi) * 0.0820 * dr * (
        ws * np.sin(lat_rad) * np.sin(delta) + np.cos(lat_rad) * np.cos(delta) * np.sin(ws)
    )
    tmean = (df["tmax_c"] + df["tmin_c"]) / 2.0
    tdiff = np.maximum(df["tmax_c"] - df["tmin_c"], 0.0)
    et0 = 0.0023 * (tmean + 17.8) * np.sqrt(tdiff) * (0.408 * ra)
    return np.round(et0, 3)


def fetch_gee_point_series(col, point_geom, scale, band_name):
    try:
        raw = col.getRegion(point_geom, scale).getInfo()
        if not raw or len(raw) <= 1 or band_name not in raw[0]:
            return pd.DataFrame(columns=["date", band_name])
        df_tmp = pd.DataFrame(raw[1:], columns=raw[0])
        df_tmp[band_name] = pd.to_numeric(df_tmp[band_name], errors="coerce")
        df_tmp = df_tmp.dropna(subset=[band_name]).copy()
        if df_tmp.empty:
            return pd.DataFrame(columns=["date", band_name])
        df_tmp["date"] = pd.to_datetime(df_tmp["time"], unit="ms").dt.normalize()
        return df_tmp[["date", band_name]]
    except Exception:
        return pd.DataFrame(columns=["date", band_name])


def run_final_pipeline():
    os.makedirs("data", exist_ok=True)

    # 1. Verificar grilla 30m
    if os.path.exists("data/grid_30m.parquet"):
        df_grid = pd.read_parquet("data/grid_30m.parquet")
        print(f"1/4 Grilla 30m verificada ({len(df_grid)} píxeles).")
    else:
        df_grid = build_grid_30m()

    # 2. Verificar series HLS
    if os.path.exists("data/hls_series_2016_2025.parquet"):
        print("2/4 Series HLS verificadas en disco.")
    else:
        build_hls_series(df_grid)

    # 3. Descargar NASA POWER (1981-2025) + respaldo físico de humedad radicular (GWETROOT)
    print("3/4 Descargando NASA POWER (1981-2025) y SMAP/GPM (2016-2025)...")
    power_url = (
        f"https://power.larc.nasa.gov/api/temporal/daily/point?"
        f"parameters=T2M_MAX,T2M_MIN,ALLSKY_SFC_SW_DWN,PRECTOTCORR,GWETROOT"
        f"&community=AG&longitude={CENTER_LON}&latitude={CENTER_LAT}"
        f"&start=19810101&end=20251231&format=JSON"
    )
    r = requests.get(power_url, timeout=60).json()
    props = r["properties"]["parameter"]

    dates = pd.to_datetime(list(props["T2M_MAX"].keys()), format="%Y%m%d")
    df_power = pd.DataFrame({
        "date": dates,
        "tmax_c": list(props["T2M_MAX"].values()),
        "tmin_c": list(props["T2M_MIN"].values()),
        "solar_rad": list(props["ALLSKY_SFC_SW_DWN"].values()),
        "precip_power_mm": list(props["PRECTOTCORR"].values()),
        "gwetroot": list(props["GWETROOT"].values()) if "GWETROOT" in props else np.nan,
    }).replace(-999.0, np.nan).ffill().bfill()

    df_power["year"] = df_power["date"].dt.year
    df_power["et0_mm"] = compute_et0_hargreaves(df_power, CENTER_LAT)

    df_power[["date", "year", "tmax_c", "tmin_c", "solar_rad", "precip_power_mm", "et0_mm"]].to_parquet(
        "data/climate_long_1981_2025.parquet", index=False
    )

    # 4. Descargar SMAP L4 (v008) y GPM IMERG sobre el centroide de la parcela
    pt_center = ee.Geometry.Point([CENTER_LON, CENTER_LAT])
    smap_col = ee.ImageCollection("NASA/SMAP/SPL4SMGP/008").select("sm_rootzone")
    gpm_col = ee.ImageCollection("NASA/GPM_L3/IMERG_V07").select("precipitation")

    smap_dfs, gpm_dfs = [], []
    for y in range(2016, 2026):
        df_sm_y = fetch_gee_point_series(smap_col.filterDate(f"{y}-01-01", f"{y+1}-01-01"), pt_center, 9000, "sm_rootzone")
        if not df_sm_y.empty:
            smap_dfs.append(df_sm_y.groupby("date", as_index=False)["sm_rootzone"].mean())

        df_gpm_y = fetch_gee_point_series(gpm_col.filterDate(f"{y}-01-01", f"{y+1}-01-01"), pt_center, 11000, "precipitation")
        if not df_gpm_y.empty:
            df_gpm_y["gpm_rain_mm"] = df_gpm_y["precipitation"] * 0.5
            gpm_dfs.append(df_gpm_y.groupby("date", as_index=False)["gpm_rain_mm"].sum())

        print(f"    Año {y} procesado.")

    # Construir tabla alineada 2016-2025 sin posibilidad de KeyError
    df_sub = df_power[df_power["year"].between(2016, 2025)].copy()
    df_aligned = df_sub[["date", "year", "tmax_c", "tmin_c", "solar_rad", "et0_mm"]].copy()

    # Unir SMAP + respaldo físico GWETROOT (escalado a porosidad típica de Iowa ~0.45 m3/m3)
    if smap_dfs:
        df_smap = pd.concat(smap_dfs, ignore_index=True).groupby("date", as_index=False)["sm_rootzone"].mean()
        df_aligned = df_aligned.merge(df_smap, on="date", how="left")
    if "sm_rootzone" not in df_aligned.columns:
        df_aligned["smap_rootzone"] = np.nan

    backup_sm = (df_sub["gwetroot"] * 0.45).values
    df_aligned["smap_rootzone"] = df_aligned["smap_rootzone"].fillna(pd.Series(backup_sm, index=df_aligned.index))
    df_aligned["smap_rootzone"] = df_aligned["smap_rootzone"].interpolate(method="linear").bfill().ffill().fillna(0.28)

    # Unir GPM + respaldo físico PRECTOTCORR
    if gpm_dfs:
        df_gpm = pd.concat(gpm_dfs, ignore_index=True).groupby("date", as_index=False)["gpm_rain_mm"].sum()
        df_aligned = df_aligned.merge(df_gpm, on="date", how="left")
    if "gpm_rain_mm" not in df_aligned.columns:
        df_aligned["gpm_rain_mm"] = np.nan

    df_aligned["gpm_rain_mm"] = df_aligned["gpm_rain_mm"].fillna(df_sub["precip_power_mm"]).fillna(0.0)
    df_aligned.to_parquet("data/aligned_2016_2025.parquet", index=False)

    # 5. Metadatos, resumen de auditoría y empaquetado ZIP
    print("\n4/4 Auditoría final de archivos generados:")
    for fname in [
        "grid_30m.parquet",
        "hls_series_2016_2025.parquet",
        "climate_long_1981_2025.parquet",
        "aligned_2016_2025.parquet",
    ]:
        df_check = pd.read_parquet(f"data/{fname}")
        print(f"    - {fname}: {df_check.shape[0]} filas x {df_check.shape[1]} columnas")

    crop_counts_2025 = df_grid["cdl_2025"].value_counts(normalize=True).round(3).to_dict()
    metadata = {
        "fecha_extraccion_utc": datetime.utcnow().isoformat(),
        "parcela_centro": {"lon": CENTER_LON, "lat": CENTER_LAT},
        "bbox": BBOX,
        "total_pixeles_agricolas_30m": int(len(df_grid)),
        "distribucion_cdl_2025": {str(k): v for k, v in crop_counts_2025.items()},
        "semilla_rng": SEED,
        "datasets": [
            "USDA/NASS/CDL (2016-2025)",
            "USDA NRCS SSURGO SDA REST",
            "USGS/SRTMGL1_003",
            "NASA/HLS/HLSL30+HLSS30 v002",
            "NASA/SMAP/SPL4SMGP/008 (con respaldo NASA POWER GWETROOT)",
            "NASA/GPM_L3/IMERG_V07 (con respaldo NASA POWER PRECTOTCORR)",
            "NASA POWER Agroclimatology (1981-2025)",
        ],
    }
    with open("data/metadata_run.json", "w") as f:
        json.dump(metadata, f, indent=2)

    shutil.make_archive("agropulse_data", "zip", "data")
    print("\n¡LISTO! Descarga 'agropulse_data.zip' desde el panel izquierdo de Colab.")

run_final_pipeline()

1/4 Grilla 30m verificada (360 píxeles).
2/4 Series HLS verificadas en disco.
3/4 Descargando NASA POWER (1981-2025) y SMAP/GPM (2016-2025)...
    Año 2016 procesado.
    Año 2017 procesado.
    Año 2018 procesado.
    Año 2019 procesado.
    Año 2020 procesado.
    Año 2021 procesado.
    Año 2022 procesado.
    Año 2023 procesado.
    Año 2024 procesado.
    Año 2025 procesado.

4/4 Auditoría final de archivos generados:
    - grid_30m.parquet: 360 filas x 19 columnas
    - hls_series_2016_2025.parquet: 452666 filas x 9 columnas
    - climate_long_1981_2025.parquet: 16436 filas x 7 columnas
    - aligned_2016_2025.parquet: 3653 filas x 8 columnas


/tmp/ipykernel_4128/617277547.py:137: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fecha_extraccion_utc": datetime.utcnow().isoformat(),



¡LISTO! Descarga 'agropulse_data.zip' desde el panel izquierdo de Colab.


In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path('/content/drive/MyDrive/agropulse_xai/data')
for f in sorted(data_dir.glob('*.parquet')):
    df = pd.read_parquet(f)
    print(f"{f.name}: {df.shape[0]} filas x {df.shape[1]} cols")

bare_soil_windows.parquet: 18 filas x 9 cols
metrics_phenology.parquet: 3600 filas x 10 cols
phenology_weekly.parquet: 507 filas x 19 cols


In [ ]:
from pathlib import Path
import pandas as pd

base = Path('/content/drive/MyDrive/agropulse_xai')

requeridos = [
    base / 'AGENT_RULES.md',
    base / 'fase1.txt',
    base / 'DETALLESNASA.txt',
    base / 'catalog' / 'crops_catalog.json',
    base / 'core' / '__init__.py',
]

print("=== ARCHIVOS BASE ===")
for r in requeridos:
    print(f"[{'OK' if r.exists() else 'FALTA'}] {r.name}")

print("\n=== DATASETS EN data/ ===")
for p in sorted((base / 'data').glob('*.parquet')):
    df = pd.read_parquet(p)
    print(f"[OK] {p.name} -> {df.shape[0]} filas x {df.shape[1]} cols")

=== ARCHIVOS BASE ===
[OK] AGENT_RULES.md
[OK] fase1.txt
[OK] DETALLESNASA.txt
[OK] crops_catalog.json
[OK] __init__.py

=== DATASETS EN data/ ===
[OK] bare_soil_windows.parquet -> 18 filas x 9 cols
[OK] metrics_phenology.parquet -> 3600 filas x 10 cols
[OK] phenology_weekly.parquet -> 507 filas x 19 cols


# **OBTENER LOS DOCUMENTOS "elevation_m" Y "slope_pct"**

In [13]:
import ee
import pandas as pd

# Esto abrirá el flujo interactivo de autenticación de Google
ee.Authenticate()

# Reemplaza 'tu-id-de-proyecto' con el ID real de tu proyecto en Google Cloud / Earth Engine
ee.Initialize(project='sebastian-borja-spaceapps')

# 1. Definir el punto central o la región de interés (Story County, Ames, IA)
roi = ee.Geometry.Point([-93.515, 42.083]).buffer(1500) # Buffer ajustado a la zona piloto

# 2. Cargar DEM (USGS 3DEP para EE. UU. o SRTM) y calcular pendiente
dem = ee.Image('USGS/3DEP/10m_collection').select('elevation')
slope = ee.Terrain.slope(dem).select('slope')

# Combinar en una sola imagen de 2 bandas
terrain_img = dem.rename('elevation_m').addBands(slope.rename('slope_pct'))

# 3. Si tienes un DataFrame con tus 360 píxeles (con pixel_id, lat, lon):
# (Aquí simulamos la carga de tus coordenadas base)
# df_pixels = pd.read_parquet("tu_grilla_coordenadas.parquet")

# Convertir puntos a un FeatureCollection de GEE para extraer valores exactos
# features = [ee.Feature(ee.Geometry.Point([row.lon, row.lat]), {'pixel_id': row.pixel_id}) for _, row in df_pixels.iterrows()]
# fc = ee.FeatureCollection(features)

# Extraer valores de la imagen raster para cada punto
# sampled_fc = terrain_img.reduceRegions(
#     collection=fc,
#     reducer=ee.Reducer.first(),
#     scale=30 # Resolución de 30 metros alineada al proyecto
# )

print("Script GEE listo para extraer elevación y pendiente acotada a la poligonal.")

Script GEE listo para extraer elevación y pendiente acotada a la poligonal.


In [17]:
from pathlib import Path

# Buscamos qué archivos hay realmente en la carpeta data y sus subcarpetas
data_dir = Path('/content/drive/MyDrive/agropulse_xai/data')
print("--- Archivos disponibles en tu estructura de Drive ---")
for file_path in data_dir.rglob('*.parquet'):
    print(file_path.relative_to(data_dir))

--- Archivos disponibles en tu estructura de Drive ---
bare_soil_windows.parquet
phenology_weekly.parquet
metrics_phenology.parquet
crop_climate_metrics.parquet
strategy_climate_metrics_long_1981_2025.parquet
whatif_grid.parquet
metrics_climate_water.parquet
strategy_climate_metrics.parquet
climate_indices_weekly.parquet
parquet_VERIFICADO/D2_ssurgo_soil.parquet
parquet_VERIFICADO/climate_long_1981_2025.parquet
parquet_VERIFICADO/hls_series_2016_2025.parquet
parquet_VERIFICADO/aligned_2016_2025.parquet
ANTIGUO(NO UTILIZAR, DATOS CRUDOS)/D7_nasa_power_daily.parquet
ANTIGUO(NO UTILIZAR, DATOS CRUDOS)/D1_D3_grid30m_cdl_srtm.parquet
ANTIGUO(NO UTILIZAR, DATOS CRUDOS)/D4_hls_ndvi_bsi_series.parquet
ANTIGUO(NO UTILIZAR, DATOS CRUDOS)/phenology_weekly.parquet
ANTIGUO(NO UTILIZAR, DATOS CRUDOS)/bare_soil_windows.parquet
ANTIGUO(NO UTILIZAR, DATOS CRUDOS)/grid_30m.parquet
ANTIGUO(NO UTILIZAR, DATOS CRUDOS)/CRUDOS/D5_D6_smap_gpm_daily.parquet


In [21]:
import ee

# Fuerza credenciales limpias con los nuevos permisos
ee.Authenticate(force=True)

# Usa el ID exacto de la columna "ID" en Google Cloud (con guiones, sin guiones bajos)
ee.Initialize(project='sebastian-borja-spaceapps')
print("✅ Conectado:", ee.String('OK').getInfo())

✅ Conectado: OK


In [29]:
import math
import ee
import pandas as pd
from pathlib import Path

# 1. Inicializar Earth Engine
ee.Initialize(project='sebastian-borja-spaceapps')

# 2. Definir rutas oficiales
DRIVE_DIR = Path('/content/drive/MyDrive/agropulse_xai/data')
INPUT_PATH = DRIVE_DIR / 'ANTIGUO(NO UTILIZAR, DATOS CRUDOS)' / 'grid_30m.parquet'
VERIFICADO_DIR = DRIVE_DIR / 'parquet_VERIFICADO'
OUTPUT_PATH = VERIFICADO_DIR / 'grid_30m.parquet'

# Limpiar el archivo con el nombre anterior para evitar duplicados en parquet_VERIFICADO
old_file = VERIFICADO_DIR / 'harmonized_grid_30m.parquet'
if old_file.exists():
    old_file.unlink()

# 3. Cargar DEM como ImageCollection y calcular elevación, pendiente en grados y en %
dem_col = ee.ImageCollection('USGS/3DEP/10m_collection').select('elevation')
proj = dem_col.first().projection()
dem = dem_col.mosaic().setDefaultProjection(proj)

slope_deg = ee.Terrain.slope(dem)
slope_pct = slope_deg.multiply(math.pi / 180.0).tan().multiply(100.0)

terrain_img = (
    dem.rename('elevation_m')
    .addBands(slope_deg.rename('slope_deg'))
    .addBands(slope_pct.rename('slope_pct'))
)

# 4. Leer grilla base y asegurar los 360 píxeles (px_0000 a px_0359)
df_grid = pd.read_parquet(INPUT_PATH)
if 'pixel_id' in df_grid.columns:
    df_grid = df_grid.sort_values('pixel_id').head(360).copy()

# 5. Extraer topografía limpia a 30m de resolución
features = [
    ee.Feature(ee.Geometry.Point([row.lon, row.lat]), {'pixel_id': row.pixel_id})
    for _, row in df_grid.iterrows()
]
fc = ee.FeatureCollection(features)

sampled_fc = terrain_img.reduceRegions(
    collection=fc,
    reducer=ee.Reducer.first(),
    scale=30
)

data_list = sampled_fc.getInfo()['features']
df_topo = pd.DataFrame([f['properties'] for f in data_list])

# 6. Fusionar topografía preservando variables físicas de suelo
df_final = df_grid.drop(columns=['elevation_m', 'slope_deg', 'slope_pct'], errors='ignore')
df_final = df_final.merge(
    df_topo[['pixel_id', 'elevation_m', 'slope_deg', 'slope_pct']],
    on='pixel_id',
    how='left'
)

# 7. Estandarización de nombres de columnas
# 7a. Cambiar prefijo cdl_ a crop_ (cdl_2016 -> crop_2016 ... crop_2025)
rename_map = {
    col: col.replace('cdl_', 'crop_', 1)
    for col in df_final.columns
    if col.startswith('cdl_')
}
# 7b. Cambiar awc a agua_disponible_awc
if 'awc' in df_final.columns:
    rename_map['awc'] = 'agua_disponible_awc'

df_final = df_final.rename(columns=rename_map)

# 8. Exportación final a parquet_VERIFICADO/grid_30m.parquet
VERIFICADO_DIR.mkdir(parents=True, exist_ok=True)
df_final.to_parquet(OUTPUT_PATH, index=False)

# 9. Auditoría rápida de verificación
expected_cols = [
    'pixel_id', 'elevation_m', 'slope_deg', 'slope_pct',
    'organic_matter', 'sand_pct', 'clay_pct', 'k_factor', 'agua_disponible_awc'
]
missing_cols = [c for c in expected_cols if c not in df_final.columns]
crop_cols = sorted([c for c in df_final.columns if c.startswith('crop_')])

print(f"✅ Archivo guardado en: {OUTPUT_PATH}")
print(f"• Registros: {len(df_final)} | Rango IDs: {df_final['pixel_id'].iloc[0]} -> {df_final['pixel_id'].iloc[-1]}")
print(f"• Columnas de cultivos ({len(crop_cols)}): {crop_cols}")
print(f"• Columnas faltantes de la lista: {missing_cols if missing_cols else 'Ninguna (100% completo)'}")

✅ Archivo guardado en: /content/drive/MyDrive/agropulse_xai/data/parquet_VERIFICADO/grid_30m.parquet
• Registros: 360 | Rango IDs: px_0000 -> px_0359
• Columnas de cultivos (10): ['crop_2016', 'crop_2017', 'crop_2018', 'crop_2019', 'crop_2020', 'crop_2021', 'crop_2022', 'crop_2023', 'crop_2024', 'crop_2025']
• Columnas faltantes de la lista: Ninguna (100% completo)
